# Baseline classifier comparison

Task #6 trains and compares baseline models (KNN, Logistic Regression, SVM, Naive Bayes) on the TF-IDF features from task 5 (`data/processed/tfidf/`) and selects the best one by **validation** macro F1. It also runs a grid search per model (hyperparameter tuning) and reports that comparison side by side with the untuned one — both are kept, and which one to treat as the baseline is left to issue #7.

This task is validation-only by design — the held-out test set, a confusion matrix, and the official "performance floor" documentation are issue #7's scope. Use the Python environment from `requirements.txt`.

In [4]:
from pathlib import Path
import importlib.util

ROOT = Path.cwd()
if not (ROOT / 'scripts/train_baseline_models.py').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'scripts/train_baseline_models.py').is_file():
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')

# Loaded by file path rather than `from scripts.train_baseline_models import ...`:
# some environments (e.g. a ROS install on PYTHONPATH) ship their own package
# named `scripts`, which would otherwise shadow this repository's scripts/ folder.
spec = importlib.util.spec_from_file_location(
    'train_baseline_models', ROOT / 'scripts/train_baseline_models.py')
train_baseline_models = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_baseline_models)
train_and_compare = train_baseline_models.train_and_compare

## Train and compare

This fits all four models on the training split at default hyperparameters, evaluates each on validation, and selects the best by macro F1 (equal weight to safe/toxic/jailbreak, rather than letting the 93% safe class dominate). It also grid-searches each model's hyperparameters (see `TUNING_GRIDS` in the script) and selects the best tuned model the same way. Both selected models' trained objects are saved, but neither is run against the test split here. Rerunning this cell replaces `data/processed/models/`.

In [5]:
data_dir = ROOT / 'data/processed/tfidf'
output_dir = ROOT / 'data/processed/models'
report = train_and_compare(data_dir, output_dir, seed=42)

print('Validation macro F1 by model:')
for name, result in report['models'].items():
    m = result['validation']['macro']
    print(f"  {name}: P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f}")

best_name = report['best_model']
best = report['models'][best_name]
print(f"\nBest model: {best_name} (validation macro F1={best['validation']['macro']['f1']:.3f})")
print('Test-set evaluation is out of scope here — see issue #7.')

/home/kange/.local/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Validation macro F1 by model:
  knn: P=0.559 R=0.396 F1=0.421
  logistic_regression: P=0.659 R=0.759 F1=0.700
  svm: P=0.832 R=0.668 F1=0.732
  naive_bayes: P=0.478 R=0.786 F1=0.522

Best model: svm (validation macro F1=0.732)
Test-set evaluation is out of scope here — see issue #7.


/home/kange/.local/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


## Per-label breakdown for the best model (validation only)

In [6]:
print(f'{best_name} — validation, per label:')
for label, metrics in best['validation']['per_label'].items():
    print(f"  {label}: P={metrics['precision']:.3f} R={metrics['recall']:.3f} "
          f"F1={metrics['f1']:.3f} (support={metrics['support']})")

svm — validation, per label:
  jailbreak: P=0.900 R=0.562 F1=0.692 (support=16)
  safe: P=0.964 R=0.989 F1=0.976 (support=696)
  toxic: P=0.633 R=0.452 F1=0.528 (support=42)


In [10]:
print('Tuned validation macro F1 by model (vs. untuned):')
for name, result in report['tuned']['models'].items():
    tuned_f1 = result['validation']['macro']['f1']
    untuned_f1 = report['models'][name]['validation']['macro']['f1']
    print(f"  {name}: untuned F1={untuned_f1:.3f}  tuned F1={tuned_f1:.3f}  "
          f"best_params={result['best_params']}")

best_tuned_name = report['tuned']['best_model']
best_tuned = report['tuned']['models'][best_tuned_name]
print(f"\nBest model (untuned): {best_name} (F1={best['validation']['macro']['f1']:.3f})")
print(f"Best model (tuned):   {best_tuned_name} (F1={best_tuned['validation']['macro']['f1']:.3f})")

Tuned validation macro F1 by model (vs. untuned):
  knn: untuned F1=0.421  tuned F1=0.678  best_params={'metric': 'cosine', 'n_neighbors': 3, 'weights': 'distance'}
  logistic_regression: untuned F1=0.700  tuned F1=0.738  best_params={'C': 10.0}
  svm: untuned F1=0.732  tuned F1=0.732  best_params={'C': 1.0}
  naive_bayes: untuned F1=0.522  tuned F1=0.657  best_params={'alpha': 0.001}

Best model (untuned): svm (F1=0.732)
Best model (tuned):   logistic_regression (F1=0.738)


In [9]:
print(f'{best_tuned_name} (tuned) — validation, per label:')
for label, metrics in best_tuned['validation']['per_label'].items():
    print(f"  {label}: P={metrics['precision']:.3f} R={metrics['recall']:.3f} "
          f"F1={metrics['f1']:.3f} (support={metrics['support']})")

logistic_regression (tuned) — validation, per label:
  jailbreak: P=0.714 R=0.625 F1=0.667 (support=16)
  safe: P=0.973 R=0.967 F1=0.970 (support=696)
  toxic: P=0.542 R=0.619 F1=0.578 (support=42)


The remaining files are `best_model.pkl` (untuned best, SVM), `best_model_tuned.pkl` (tuned best), both loadable with `pickle.load`, and `model_comparison.json` (every model's full validation metrics, best hyperparameters, and the full per-combination grid search results, for both the untuned and tuned comparisons).

See [the baseline classifier guide](../docs/baseline-classifier.md) for the full comparison tables, why macro F1 is the selection metric, why these four models and this tuning approach were chosen, and model-specific limitations. 

Neither model here has been evaluated on test, and the choice between the untuned and tuned baseline is deliberately left open — test-set evaluation against the project's 80%+ F1 target, a confusion matrix, and the official baseline-floor documentation are issue #7's scope, not this notebook's.